# Lab 2 — Getting data in
### EP0418 · Topic 2 · 2 practical hours (Week 2)

**You need:** `sg_rainfall_daily.csv`, `rain_now.json` and `ep0418.py`, which the
notebook fetches, and the printed Lab 2 sheet (the PDF under Topic 2 on
Brightspace; your tutor hands out copies). The sheet has boxes for what you write
down; the notebook has the same boxes if you prefer to type.

---
Singapore has about ninety rain gauges, and the government publishes what each one
measured every five minutes. Today that data arrives in the three ways the Topic 2B
notes describe, and you meet all three:

- **a CSV**: a saved table, sixty days of rain, one row per day;
- **JSON**: what the live service actually sends, boxes inside boxes, one
  five-minute snapshot of every gauge;
- **a live API**: the service itself, which will surprise you.

Then you do what Topic 2A is about: **plan first**, then prompt, generate, verify,
refine, to turn the JSON into a table you can ask a question of. Each task below
says which part of Topic 2 it practises. This lab is the same weight as Lab 1.

## How the cells work

- **Run every grey code cell when you reach it**, top to bottom: click in it and
  press Shift+Enter. A step tells you if a cell needs something first.
- **Text boxes** are the cells with a bold label ending in a colon, like
  **My answers:**. Double-click one to type in it, then press Shift+Enter. You may
  write on the printed sheet instead.
- **Getting code from Gemini.** Each task gives you a prompt to paste into Gemini.
  Gemini puts its code into the notebook as a **green cell**: click **Accept and
  run** (bottom of the chat panel). Then run the check cell under the task. Its
  first line, starting `# TODO(you):`, says what must run first. Lines like
  `s = keep("s")` pick up what Gemini's cell made: do not edit them. If the check
  cell says *Not there yet* (or *No ... yet*), Gemini used a different name: type in
  the chat *call the result* followed by the name the task asks for, accept the new
  cell, and run the check cell again.
- **Tables to fill in**: double-click the table and it shows lines of `|` and `---`.
  Keep every `|` and the `|---|` line; type each answer between two `|` marks in the
  right row, then press Shift+Enter to see the table again.
- **Fill-in cells** have lines like `TEAM = ""` or `H = None` for you to type into:
  type text between the two quote marks (`TEAM = "Team Rocket"`), or replace `None`
  with a number (`H = 6`), keeping the rest of the line. Type numbers
  without commas (`128000`, not `128,000`). An example is always shown.

---
## How every lab in this module works

You will **not** write this code from scratch. Ask your AI assistant to write it.
Your job is the part it cannot do: deciding what the right answer looks like,
reading what it gave you, and proving it did what you asked.

Every prompt in these notebooks opens with the rules that matter for that task,
such as do not drop rows, show the row count before and after, or explain the code
in plain words. Keep those lines when you adapt a prompt, and start your own
prompts the same way.

At the foot of each task is a **verification contract** — a few `check.*` lines.
**You may not edit them.** The AI can write any implementation it likes; it cannot
satisfy these by accident.

If a check fails, read the message. It tells you what was found and what to try
next. Failing checks are the normal state of a lab in progress, not a sign you
are behind.

> **First: File → Save a copy in Drive, and work in your copy.** This page opened from GitHub, and changes here are not saved. Reopen your copy from Google Drive next time.

> **Then run the grey code cell just below** (click in it and press Shift+Enter, or click the play button at its left). Colab may warn that the notebook was not authored by Google: click **Run anyway**. You should see nothing in red; lines like *fetched ...*, a row count or a short table are normal. That means the course files and tools are loaded. Every lab starts this way. From here on, **run every grey code cell when you reach it**, top to bottom. **To write in a text box** (a cell with a bold label ending in a colon), double-click it, type, then press Shift+Enter.

In [ ]:
import os, urllib.request
REPO = "https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/"


def fetch(*names):
    """Download any course file that is not beside the notebook yet."""
    for name in names:
        if os.path.exists(name):
            continue
        try:
            urllib.request.urlretrieve(REPO + name, name)
            print("fetched", name)
        except Exception:
            print(f"Could not download {name}. Upload it with the folder icon on the left.")
            from google.colab import files
            files.upload()


fetch("ep0418.py")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from ep0418 import check, show


def keep(name):
    """The value Gemini's cell already made under this name, or ... if there is none yet."""
    if name not in globals():
        print(f"Not there yet: nothing called {name}. If Gemini's green cell has run, type in the "
              f"same chat: call the result {name}. Then Accept and run, and run this cell again.")
    return globals().get(name, ...)


def filled(*values):
    """True once every TODO above has a value. Otherwise says what to do."""
    if any(v is ... or v is None for v in values):
        print("Not there yet. Run Gemini's code first (click Accept and run on its green cell, "
              "or paste its code into a new cell and run it), then run this cell again.")
        return False
    return True


def submit_prediction(team, lab, prediction, method):
    """Send one team's number to the Prediction League board."""
    import urllib.parse
    form = "https://docs.google.com/forms/d/e/1FAIpQLSdwpzHafC_rmPDdbK_cm6J8CihD-NSVS7ZBUyYCWIrS_MMJ0g/formResponse"
    fields = {'team': 'entry.911171418', 'lab': 'entry.1890998159', 'prediction': 'entry.959891957', 'method': 'entry.1997288724'}
    if not team or not method or prediction is None:
        print("Fill in TEAM, MY_PREDICTION and METHOD above, then run this cell again."); return
    entry = {"team": team, "lab": str(lab), "prediction": f"{float(prediction):.1f}", "method": method}
    if not form:
        print("The Prediction League is not switched on yet; tell your tutor your number."); return
    data = urllib.parse.urlencode({fields[k]: v for k, v in entry.items()}).encode()
    try:
        urllib.request.urlopen(urllib.request.Request(form, data=data, headers={"User-Agent": "EP0418"}), timeout=60)
        print(f"Entered: {team} predicts {float(prediction):.1f}. Check the scoreboard in a minute.")
    except Exception as e:
        print("The league did not answer:", e, "- try again in a minute.")

import json
fetch("sg_rainfall_daily.csv", "rain_now.json")

## Task 1 — A saved table (CSV), and the five questions (20 minutes)

*Topic 2B: the five questions to ask of any data you have never seen.*

A CSV is a table saved as text. The notes give five questions to ask of any data
you have never seen. The cell asks them for you, in order.

1. **Run the cell below.** It prints one line per question.
2. **Read every line**, then the chart under them.
3. **Write down** the strangest number in the text box labelled *Strangest
   number:* below the chart. "What you should see" under the chart tells you what
   is normal.

In [ ]:
rain = pd.read_csv("sg_rainfall_daily.csv", parse_dates=["date"])

print("1. Rows and columns:", rain.shape, list(rain.columns))
print("2. Period:", rain.date.min().date(), "to", rain.date.max().date(),
      "| largest gap between rows:", rain.date.diff().max().days, "day(s)")
print("3. Smallest and largest, each column:")
print(rain.describe().loc[["min", "max"]].round(2))
print("4. Blanks:", int(rain.isna().sum().sum()), "| days with zero rain everywhere:", int((rain.rain_mm_mean == 0).sum()))
print("5. Anything impossible? negative rain:", int((rain[["rain_mm_mean", "rain_mm_max"]] < 0).sum().sum()))
show.series(rain, col="rain_mm_mean", title="Average rain across all gauges, mm per day")

**Expected output** *(computed from the data file when this notebook was built)*: Line 1: 60 rows, four columns. Line 2: every day present, largest gap 1 day.
Line 3: a small table of each column's smallest and largest value. Line 4: no blanks; 3 completely dry days. Line 5: no negative rain.

**What you should see.** Sixty days, no gaps, no blanks. Now look at the "max"
line: the wettest single reading is far above the daily average. Rain in
Singapore falls hard in one place for twenty minutes while the rest of the island
is dry. And `stations_reporting` is not the same every day.

**Write it down:** the strangest number on screen, and what you think caused it.

**Strangest number:**

<details><summary><b>Answer guidance</b> (open after you have written yours)</summary>

Two good answers. The largest `rain_mm_max` is enormous next to the average,
because it is one gauge under one storm cloud while the average is spread over
ninety gauges, most of them dry. And `stations_reporting` moves between two values:
on some days one gauge sent nothing. That is not an error in the file; it is a
gauge that was offline, and it is the first hint that "missing" and "zero" are
different things. Lab 4 is about that difference.
</details>

## Task 2 — Boxes inside boxes (JSON), and the live feed (20 minutes)

*Topic 2B: JSON is not a table, and a live API will surprise you.*

`rain_now.json` is one five-minute snapshot, saved exactly as the live service sent
it.

1. **Run the cell below.** It opens the file and prints its shape: the boxes, then
   what is inside the first box of each kind.
2. **Read the shape before the numbers.** Count the stations and count the values.
3. **Then run the live cell** further down, once, and read whatever it says.

In [ ]:
raw = json.load(open("rain_now.json"))["data"]

print("top-level boxes:", list(raw.keys()))
print("how many stations:", len(raw["stations"]), "| first one:", raw["stations"][0])
reading = raw["readings"][0]
print("the reading is from:", reading["timestamp"])
print("how many values in it:", len(reading["data"]), "| first three:", reading["data"][:3])

**What you should see.** Two lists that belong together but are stored apart: the
stations (id, name, location) and the readings (station id, value). Count them:
there are more stations than values. Two gauges sent nothing in this snapshot. To
answer "which gauge was wettest" you have to join the lists by station id without
losing those two, and without pretending they measured zero. That is the work
Task 3 asks Gemini to do.

**Now the live service.** The cell below asks for the snapshot being taken right
now. Run it once. It may work; it may answer 403 (refused, no reason given) or 429
(too many requests, forty of you at once). Either way, read the message: the notes
list exactly these three surprises, and you have just met one. Nothing later
depends on this cell.

In [ ]:
try:
    req = urllib.request.Request("https://api-open.data.gov.sg/v2/real-time/api/rainfall",
                                 headers={"User-Agent": "EP0418 lab"})
    live = json.load(urllib.request.urlopen(req, timeout=20))["data"]
    vals = [x["value"] for x in live["readings"][0]["data"]]
    print("live:", live["readings"][0]["timestamp"], "|", len(vals), "gauges |",
          sum(v > 0 for v in vals), "reporting rain right now | wettest", max(vals), "mm")
except Exception as e:
    print("The service surprised you:", e)
    print("That is normal (see the notes: pagination, rate limits, silent rejection). "
          "Build on the saved file; demonstrate on the feed.")

## Task 3 — Plan first, then the loop (45 minutes)

*Topic 2A: plan, prompt, generate, verify, refine. "It ran" is the weakest evidence.*

**The goal.** Right now the gauge names are in one list and the rain values in
another, and nothing joins them except the station id. You want one table that
puts them side by side: one row for each gauge, with two columns, `name` (where
the gauge is) and `rain_mm` (how much rain it measured in those five minutes).
Like this:

| name | rain_mm |
|---|---|
| Bukit Batok Street 34 | 0.0 |
| Upper Changi Road North | 2.4 |
| ... | ... |

With that table, "which ten gauges were wettest?" is one line.

**Step 1. Ask for a plan, no code.** Open Gemini with the **spark icon** at the
bottom centre of the Colab window, click **+** for a new chat, paste this and press Enter (to copy it, drag across the quoted text from its first word to its last and press Ctrl+C, Cmd+C on a Mac):

> I am new to this. Answer in three plain sentences, no headings, no bullet
> points, no code. I have JSON from a rain service with two lists: "stations"
> (each has id and name) and "readings" (each has a timestamp and a list of
> stationId and value). How would you turn one reading into a table with one row
> per station, showing its name and its value? What could go wrong?

Read the answer. It should say: match the two lists by station id, and watch out
for stations with no reading. If it does not mention matching by id, ask *how do
you know which name goes with which value?* If it writes an essay, type *simpler,
three sentences*. Ignore the blue suggestion chips under the answer.

**Step 2. Ask for the code, in the same chat.** Copy this the same way, paste it
and press Enter. The
rules come first; `raw` already exists in Colab, so Gemini can refer to it:

> Do not drop any rows. Show the row count before and after. Explain in plain
> words what the code does. `raw` is a Python dict with `raw["stations"]`, a list
> of dicts with keys `id` and `name`, and `raw["readings"][0]["data"]`, a list of
> dicts with keys `stationId` and `value`. Write pandas code that builds a
> DataFrame `table` with one row per station and two columns, `name` and
> `rain_mm`, matching each value to its station by id. A station with no value
> must keep its row, with `rain_mm` left blank (NaN), not 0.

**Step 3. Run Gemini's code, then the check cell.** Gemini usually offers its
code as a **green cell** inserted into the notebook, with **Accept and run** at the
bottom of the chat: click that. If instead the code only appears in the chat,
click the copy icon on its code block, click once in this text cell so it is
selected, click **+ Code** at the top of the page (a new empty cell appears just
below this text), paste, and run it. Either way `table` now exists. If Gemini shows
the first few rows and they are all 0.0, that is normal: rain falls in one part of
the island at a time, and most gauges are dry in any snapshot. The wettest ten
come two cells down. Now run the check cell: the grey cell just under this text that starts
`# TODO(you): run Gemini's code first`. Read the three checks and the printed line
under them.

**Step 4. Only if a check fails:** go back to the chat and tell Gemini which check
and what it said, for example *rows_at_least failed: 75 rows, expected 77*. Accept
its new cell, run it, run the check cell again. That is the refine step. If every
check passed, there is nothing to refine; move on.

In [ ]:
# TODO(you): run Gemini's code first (its green cell above, or a new cell), so that `table` exists. Then run this cell.

# ---- verification contract — do not edit ----
if "table" not in globals() or table is None:
    print("No `table` yet. Accept and run Gemini's cell (or paste its code into a new cell and run it), then run this cell again. "
          "If you did, Gemini used another name: type in the chat 'call the result table', accept, and run this cell again.")
else:
    check.columns_present(table, ["name", "rain_mm"])
    check.rows_at_least(table, len(raw["stations"]))      # one row per station: a join that lost the silent gauges fails here
    check.values_in_range(table, "rain_mm", 0, 200)
    check.report()
    print("stations:", len(raw["stations"]), "| values in the reading:", len(raw["readings"][0]["data"]),
          "| rows in your table:", len(table), "| blank rain_mm:", int(table.rain_mm.isna().sum()),
          "| zero rain_mm:", int((table.rain_mm == 0).sum()))

**Read the last printed line**, the one that starts `stations:`. Compare your
numbers with these three possibilities:

| rows in your table | blank rain_mm | verdict |
|---|---|---|
| 77 | 2 | **Right.** The two silent gauges are in the table, marked unknown. |
| 75 | 0 | Wrong. The join quietly dropped them. `rows_at_least` fails. |
| 77 | 0 | Wrong, and **no check catches it**: "no reading" became "no rain". |

If you got the first, there is nothing to refine: skip to the hand check. The zero
count (62 or so) is real; most gauges were dry in that snapshot.

**How to refine, if you got the second or third.** Refine means telling Gemini, in
the same chat, exactly what you observed and what you expected, in numbers:

1. Click into the Gemini chat box and type one specific sentence, for example:
   *The table has 77 rows but 0 blanks. The two stations with no reading must
   have rain_mm left as NaN, not 0. Fix that and keep everything else.* Press Enter.
2. Gemini offers a new green cell. Click **Accept and run**.
3. Run the check cell again and read the last line. It should now say 2 blanks.

"It didn't work" gets you a random different attempt. "77 rows, 0 blanks, the
silent stations must be NaN" gets you a fix. That sentence is the whole of Lab 4
in one line.

**Check two rows by hand.** The notes say verify means four things: did it run, is
the shape right, is the range right, and do two rows checked by hand agree. The
checks did the first three. The fourth means: take a station, find its number in
the raw JSON, and see whether your table shows the same number against that
station's name. The cell below does the looking up for you; your job is to read
its verdict. If they agree, the join paired names and values
correctly. If they differ, the join is wrong and no check would have said so.

**Run the cell below.** It does the check for two stations: the first in the
list, and the wettest one, because a match on 0.0 proves little. Each line ends in
`match: True` or `match: False`. Read both.

In [ ]:
if "table" in globals() and table is not None:
    readings = raw["readings"][0]["data"]
    names = {st["id"]: st["name"] for st in raw["stations"]}
    wettest_id = max(readings, key=lambda x: x["value"])["stationId"]
    for sid in [raw["stations"][0]["id"], wettest_id]:          # row 1: the first station; row 2: the wettest
        raw_value = [x["value"] for x in readings if x["stationId"] == sid]
        in_table = table.loc[table.name == names[sid], "rain_mm"].tolist()
        print(f"{names[sid]:28s} raw JSON says {raw_value}   your table says {in_table}   match: {raw_value == in_table}")
    print()
    print("Wettest ten gauges in this snapshot:")
    top = table.nlargest(10, "rain_mm")
    print(top.to_string(index=False))
    top.iloc[::-1].plot.barh(x="name", y="rain_mm", legend=False, figsize=(7, 4),
                             title="Rain in the last five minutes, mm"); plt.tight_layout(); plt.show()

**Write it down:** did both hand-checked rows say `match: True`? How many gauges
had any rain at all (from the check cell's last line: *values in the reading* minus
*zero rain_mm*, for example 75 − 62 = 13)? Where was the wettest one?

**Hand check and wettest gauge:**

<details><summary><b>Answer guidance</b> (open after you have written yours)</summary>

The hand-checked value and the table value must be identical; if they are not, the
join matched the wrong id to the wrong name, which is the classic silent fault in
this task, and no check above would have caught it. In this snapshot (a morning in
late May) about a dozen of the 77 gauges show rain, the wettest just under 6 mm in
five minutes, clustered in one part of the island: a storm is small and the island
is not. Two gauges have no value at all; they must appear as blanks, not zeros.
</details>

---
## Predict: How much rain next week? (10 minutes)

*The Prediction League: one question per lab, settled by reality a week later. Teams of two or three; pick a team name you will keep all semester.*

**The question:** How much rain, in mm, will Singapore's gauges average over the seven days of the target week? One number per team.
**Teams:** turn to one or two neighbours now and agree a
name; keep the same team and name every week. **The scoreboard** is at
https://stanleysp-eng.github.io/EP0418-labs/ (open it on a phone): it shows the
target week's dates, every team's number the moment it is sent, and the real
number when the week is over. Nearest team wins the week.

**Step 1. Run the cell below.** It prints two candidate numbers, A and B, and the
exact question to ask Gemini.

In [ ]:
fetch("sg_rainfall_daily.csv")
rain = pd.read_csv("sg_rainfall_daily.csv", parse_dates=["date"])
A = rain.tail(7).rain_mm_mean.sum()            # last week, summed
B = rain.rain_mm_mean.mean() * 7               # a typical week: sixty-day average x 7
print(f"A. last week was like this        : {A:6.1f} mm   <- the baseline on the board")
print(f"B. a typical week in the file     : {B:6.1f} mm")
print(f"   (the wettest week in the file was {rain.rain_mm_mean.rolling(7).sum().max():.1f} mm, the driest {rain.rain_mm_mean.rolling(7).sum().min():.1f} mm)")

print()
print("Step 2. Open Gemini (spark icon, bottom centre), press + for a new chat, and paste this:")
print()
print(f"I am new to this. Answer in three plain sentences, no headings, no code. I must predict the island-average rain over the target week, in mm. "
      f"One estimate says {A:.0f}, based on last time; another says {B:.0f}, based on what is typical in my data. "
      "Which should I use, or a number in between, and why might it still be wrong?")

**Step 2. Ask Gemini.** Copy the printed question: drag across it in the output,
from *I am new* to *wrong?*, and press Ctrl+C (Cmd+C on a Mac). Open Gemini, click
**+** for a new chat, paste it and press Enter.
Read the three sentences. It will pick A, B or something between and give a reason.
Ignore the blue suggestion chips under its answer. If it writes an essay, type
*simpler, three sentences*.

**Step 3. Type your number and send it.** Talk it over with your team: Gemini's
pick, or your own. **One person per team** puts it in the cell below, with the team
name and one line on why, and runs the cell; the others skip that cell. Filled in, it looks like this:

```
TEAM = "Team Rocket"
MY_PREDICTION = 32
METHOD = "Gemini said B; we nudged it because ..."
```

The message *Entered: Team Rocket predicts ...* confirms it. Open the scoreboard
(https://stanleysp-eng.github.io/EP0418-labs/) and look for your team under
"Prediction League" within a minute. Nobody is marked on the method; only reality
marks the number.

In [ ]:
TEAM = ""                 # TODO(you): your team name
MY_PREDICTION = None      # TODO(you): your number, no commas, e.g. 32
METHOD = ""               # TODO(you): one line on how you chose it

submit_prediction(TEAM, 2, MY_PREDICTION, METHOD)

**Write it down** (on the sheet, or double-click here): your number, the baseline,
and the one reason you moved away from the baseline, if you did. Next lab, the
real number is on the board.

**Our number / the baseline / why:**

---
## Read before you run

An assistant was asked which rain gauge was wettest in the snapshot. It wrote the code below, which ran and printed a station name.

The code is in the cell below, commented out. It came from an AI assistant in
someone else's chat; you are not asking Gemini for it.

1. **Read it. Do not run it.** Answer questions 1 and 2 (just below) in the text
   box under the code cell (double-click it to type), or on the sheet.
2. **Ask Gemini to explain it.** Click inside the code cell, press Ctrl+A (Cmd+A on
   a Mac; inside a code cell this selects only that cell) and Ctrl+C to copy it.
   Open Gemini (spark icon, bottom centre), click **+** for a new chat, paste the
   code into the chat box, then type a space and *Explain this code in plain words.*
   after it on the same line. Press Enter once, to send. If Gemini adds a green cell,
   click **Cancel**: this step only explains. Read the explanation. Did it point out a problem with the code, the one your questions are about? It often does not.
3. **Only then run it.** Delete the `# ` at the start of every line *below the
   first line* (the first line is the instruction and stays), then run the cell
   and see whether you were right.

**Question 1.** Which row does `.iloc[0]` pick after the sort: the smallest value or the largest? What will it print on a snapshot where most gauges read zero?

**Question 2.** Predict what the code will print, and write what it should print (compare with the wettest-ten list from Task 3).

<details><summary>Hint 1: where to look</summary>

Which way does `sort_values` sort unless you tell it otherwise: smallest first, or largest first?

</details>
<details><summary>Hint 2: closer to the answer</summary>

Smallest first. Row 0 after the sort is the driest gauge, not the wettest. The code runs, prints a real station name, and is wrong. Two rows checked by hand would show it.

</details>

An explanation that reads well is not evidence the code is right; that is the
point of step 2. 

**Optional, to fix it:** in the same Gemini chat, type *Fix the problem in this
code that Hint 2 describes. Say in one sentence what you changed.* (Gemini can read
the hint in the notebook; you can open it yourself by clicking its title.) Press Enter. If Gemini's sentence names only the change you asked for,
click **Accept and run**. If it names other changes too, click **Cancel** and ask
again with *change only that, nothing else*.

In [ ]:
# ---- Every line below this one is the code. Do steps 1 and 2 first. For step 3, delete the "# " at the start of each line below, then run the cell. ----
# wettest = table.sort_values("rain_mm").iloc[0]
# print("wettest gauge:", wettest["name"], wettest["rain_mm"], "mm")

**Answer to question 1:**

**Answer to question 2:**

**Did Gemini's explanation mention the fault? (yes / no, and the sentence):**

**Was I right, once I ran it?**

---
## Quiz yourself before you leave (5 minutes)

The checks tell you whether the code is right. They do not tell you whether
**you** know what it did. In this notebook, open Gemini (spark icon, bottom centre)
and click **+** for a new chat. Copy the quoted prompt below: drag from its first
word to its last and press Ctrl+C (Cmd+C on a Mac). Paste it into the chat box and
press Enter:

> Quiz me on this lab. Ask five short questions, one at a time, in plain words, on these ideas: the three ways data arrives (CSV, JSON, live API) and what goes wrong with each; the five questions to ask of any new dataset, in order; why JSON is not a table and what joining two lists by id means; the four steps of the loop (prompt, generate, verify, refine) and the step most people skip; why 'it ran without an error' is weak evidence, and what checking two rows by hand adds. Take the examples from the code and outputs in this notebook. Wait for my answer. Then say in one sentence whether I am right, and why. Do not give the next question until I answer. After the fifth, give my score out of 5 and the one idea I should revise, in one line.

Gemini asks in the chat panel; you answer in the chat box; it marks you and asks
the next. Ignore the suggestion chips under its messages. Answer in your own
words, without looking back until it has marked you.

Then write your score and the idea to revise below. Your tutor asks for them in
the plenary.

**Quiz score:** ___ / 5

**The idea I should revise, in my own words:**

---
## Before you leave

Write these on the last page of the Lab 2 sheet, or double-click this cell and
type them here:

- The strangest number in the CSV, and what caused it.
- Which API surprise you met, if any.
- One sentence: did the code Gemini wrote in Task 3 do what its three-sentence plan
  said (matching the lists by station id, keeping the silent gauges)? Name one
  thing the plan told you to check.

**Mistake caught:** one line about any mistake the AI made today and how you saw
it. If Gemini got everything right in Task 3, use the read-before-you-run code:
*I expected the wettest gauge · it printed the driest · the sort was smallest
first.* Format: *(what I expected · what was there · how I found it)*